# 03 — Privacy Controls (Governance Evidence)

This notebook provides **governance-grade control evidence** for privacy-by-design in the data pipeline.

Evidence produced:
- **PII inventory & classification** (PII vs quasi-identifiers/proxies).
- **Data minimisation control** for the modelling dataset (`applications_analysis.csv`).
- **Pseudonymisation for linkage** using `applicant_pseudo_id`.
- **Auditability & provenance finding** (processing timestamp missing evidence).
- **Identity Resolution Governance (Duplicates & Conflicts)**
    - `duplicate_id_report.csv` provides an identity-resolution log:
        - duplicates are classified as **exact / versioned / conflicting**;
        - canonical record selection is **deterministic** to support reproducibility.
- **Proxy Risk & Fairness Governance** (spending/location/age proxies)
Even without direct PII, certain features may act as proxies for protected characteristics (e.g., location/ZIP, spending patterns, age-derived variables).  
Guardrails:
- **Prohibited:** direct PII fields in modelling datasets.
- **Conditional:** quasi-identifiers/proxies permitted only with documented purpose + monitoring.
- **Required monitoring:** disparate impact slices, drift checks, periodic governance review, and clear documentation in model cards/datasheets.

- **Summary text** ready to paste into the README.

# 0 Setup
- Imports & environment  
- Load paths and datasets

In [1]:
# Autoreload setup
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
import sys
import hashlib
from datetime import datetime

# Ensure the project root (containing src/__init__.py) is on sys.path
cwd = Path.cwd().resolve()
project_root = None
for p in [cwd, *cwd.parents]:
    if (p / "src" / "__init__.py").exists():
        project_root = p
        break

if project_root is None:
    raise FileNotFoundError("Could not find project root containing src/__init__.py")

sys.path.insert(0, str(project_root))
print("Project root added to sys.path:", project_root)

from src.governance import (
    default_privacy_paths,
    write_data_quality_report_postclean,  
    normalize_pii_inventory,
    build_pii_presence_table,
    assert_no_direct_pii_in_analysis,
    heuristic_suspicious_columns,
    safe_preview_curated,
    demo_pseudonymise,
    extract_processing_timestamp_rows,
)

Project root added to sys.path: C:\Users\anton\OneDrive - Nova SBE\Nova\S2\DEGO\DEGO_GP\DEGO_Project_Group03


In [2]:
# Resolve standard project paths 
paths = default_privacy_paths()

# Load core datasets
pii = pd.read_csv(paths.pii_inventory)
analysis = pd.read_csv(paths.applications_analysis)
curated = pd.read_csv(paths.applications_curated_full)

# Ensure post-clean DQ report exists; generate if missing
if not paths.dq_postclean.exists():
    print("DQ post-clean report not found. Generating it now...")
    write_data_quality_report_postclean(
        applications_curated_full_path=paths.applications_curated_full,
        output_path=paths.dq_postclean,
    )

dq_post = pd.read_csv(paths.dq_postclean)

# Quick shape check 
print("PII inventory:", pii.shape)
print("Analysis dataset:", analysis.shape)
print("Curated full dataset:", curated.shape)
print("DQ post-clean report:", dq_post.shape)

PII inventory: (10, 5)
Analysis dataset: (500, 16)
Curated full dataset: (502, 46)
DQ post-clean report: (26, 17)


## 1. Privacy Controls

### 1.1 PII Inventory & Classification

**Control objective:** Maintain an explicit inventory of personal data fields and their presence by data layer  
(raw / curated / analysis), enabling access control decisions and GDPR accountability

In [3]:
# Normalize inventory and extract PII / quasi-identifiers based on YOUR labels
inv, direct_fields, quasi_fields, inv_map = normalize_pii_inventory(pii)

# NOTE:
# Your inventory uses labels like "pii" and "quasi-pii" (not "direct" / "quasi").
# So we compute field lists from inv["_class"] values.
pii_fields = sorted(inv.loc[inv["_class"].eq("pii"), "_field"].unique().tolist())
quasi_pii_fields = sorted(inv.loc[inv["_class"].eq("quasi-pii"), "_field"].unique().tolist())
non_pii_fields = sorted(inv.loc[inv["_class"].eq("non-pii"), "_field"].unique().tolist())

print(f"PII fields (n={len(pii_fields)}):")
print(pii_fields)

print(f"\nQuasi-PII / proxy candidates (n={len(quasi_pii_fields)}):")
print(quasi_pii_fields)

print(f"\nNon-PII fields (n={len(non_pii_fields)}):")
print(non_pii_fields)

print("\nInventory schema mapping:", inv_map)

# Build the full "presence by layer" table
presence_full = build_pii_presence_table(
    inv=inv,
    inv_map=inv_map,
    analysis_cols=set(map(str, analysis.columns)),
    curated_cols=set(map(str, curated.columns)),
)

# keep only the most relevant columns and group by PII class
presence = (
    presence_full[["field", "pii_class", "raw", "curated", "analysis"]]
    .sort_values(["pii_class", "field"])
    .reset_index(drop=True)
)

display(presence[presence["pii_class"].isin(["pii", "quasi-pii"])])

PII fields (n=5):
['applicant_info.date_of_birth', 'applicant_info.email', 'applicant_info.full_name', 'applicant_info.ip_address', 'applicant_info.ssn']

Quasi-PII / proxy candidates (n=4):
['applicant_info.gender', 'applicant_info.zip_code', 'applicant_pseudo_id', 'application_id']

Non-PII fields (n=1):
['age_band']

Inventory schema mapping: {'field': 'field_path', 'class': 'classification', 'raw': 'present_in_raw', 'curated': 'present_in_curated', 'analysis': 'present_in_analysis'}


,field,pii_class,raw,curated,analysis
1,applicant_info.date_of_birth,pii,True,True,False
2,applicant_info.email,pii,True,True,False
3,applicant_info.full_name,pii,True,True,False
4,applicant_info.ip_address,pii,True,True,False
5,applicant_info.ssn,pii,True,True,False
6,applicant_info.gender,quasi-pii,True,True,True
7,applicant_info.zip_code,quasi-pii,True,True,True
8,applicant_pseudo_id,quasi-pii,False,False,True
9,application_id,quasi-pii,True,True,True


### 1.2 Data Minimisation Control (Model-Safe Dataset)

**Control objective:** Ensure the modelling dataset (`applications_analysis.csv`) contains **no direct identifiers**  
(e.g., name, email, SSN, IP address, DOB). PII may exist in curated/audit layers, but must remain restricted.

In [4]:
# Hard control: no PII fields should appear in the analysis dataset
# (We enforce based on the inventory label "pii")
analysis_cols = set(map(str, analysis.columns))
pii_in_analysis = sorted(set(pii_fields).intersection(analysis_cols))

print("PII fields present in analysis:", pii_in_analysis)
assert len(pii_in_analysis) == 0, (
    "DATA MINIMISATION FAILURE: PII fields found in applications_analysis.csv: "
    + ", ".join(pii_in_analysis)
)

print("Data minimisation control PASSED: No PII fields in applications_analysis.csv")

# Governance note: quasi-identifiers may remain (e.g., ZIP, gender, age bands) and require monitoring
quasi_in_analysis = sorted(set(quasi_pii_fields).intersection(analysis_cols))
print("\nQuasi-PII/proxy fields present in analysis (expected, monitor):")
print(quasi_in_analysis)

# Defence-in-depth: heuristic scan for suspicious identifier-like columns
sus = heuristic_suspicious_columns(analysis)
print("Heuristic suspicious columns in analysis (review if non-empty):")
print(sus)

PII fields present in analysis: []
Data minimisation control PASSED: No PII fields in applications_analysis.csv

Quasi-PII/proxy fields present in analysis (expected, monitor):
['applicant_pseudo_id', 'application_id']
Heuristic suspicious columns in analysis (review if non-empty):
['clean_zip_code']


### 1.3 Pseudonymisation for Linkage (`applicant_pseudo_id`)

**Control objective:** Enable deterministic linkage across records without exposing direct identifiers.

**Governance policy (narrative):**
- The modelling dataset uses a pseudonymous key (`applicant_pseudo_id`).
- The salt/secret used to generate pseudonyms must be stored in a **secrets manager**.
- Analysts must not have access to salt/mapping logic (**separation of duties + least privilege**).


In [5]:
assert "applicant_pseudo_id" in analysis.columns, "Missing applicant_pseudo_id in analysis dataset"
display(analysis[["applicant_pseudo_id"]].head())

,applicant_pseudo_id
0,fc4fb76803a008529455aa4130a4c9f4a5f72f06f7ad43...
1,7fa4238022da5aed441f8c48a907a8f3cbe88186049c8e...
2,e626311f310f7fb80415229777be761b877b33d42ddcc1...
3,417094dc0567f442dbc01b8d7007ea237a13ba958b0d77...
4,61c112b16ecefe0dab71d98f848daab97cb0019083597a...


In [6]:
# Minimal pseudonymisation demo (educational only; real salt must be stored securely)
demo_salt = "DEMO_ONLY_DO_NOT_USE_IN_PROD"
example = ["ana@example.com", "ANA@example.com", None]

display(pd.DataFrame({
    "raw_email": example,
    "pseudo_id_demo": [demo_pseudonymise(v, demo_salt) for v in example],
}))

# Safe curated preview (masked PII fields)
display(safe_preview_curated(curated, n=10))

,raw_email,pseudo_id_demo
0,ana@example.com,f9348db3c3030dd33da32ea014a59a86a0c678c7bf58d4...
1,ANA@example.com,f9348db3c3030dd33da32ea014a59a86a0c678c7bf58d4...
2,None,<NA>


,application_row_id,application_id,raw_processing_timestamp,raw_applicant_full_name,raw_applicant_email,raw_applicant_ssn,raw_applicant_ip_address,raw_applicant_gender,raw_applicant_date_of_birth,raw_applicant_zip_code,...,clean_savings_balance,clean_loan_approved,clean_interest_rate,clean_approved_amount,clean_rejection_reason,approved_missing_terms_flag,rejected_missing_reason_flag,is_duplicate_id,is_canonical_for_analysis,has_conflict
0,0,app_200,2024-01-15T00:00:00Z,***ith,***com,***340,***155,Male,***-09,10036.0,...,31212.0,False,NaN,NaN,algorithm_risk_score,False,False,False,True,False
1,1,app_037,NaN,***ker,***com,***784,***112,M,***-31,10032.0,...,17915.0,False,NaN,NaN,algorithm_risk_score,False,False,False,True,False
2,2,app_215,NaN,***ore,***com,***178,***250,Male,***-24,10075.0,...,37909.0,True,3.7,59000.0,NaN,False,False,False,True,False
3,3,app_024,NaN,***Lee,***com,***833,***.67,Male,***-25,10077.0,...,0.0,True,4.3,34000.0,NaN,False,False,False,True,False
4,4,app_184,2024-01-15T00:00:00Z,***uez,***com,***475,***105,M,***-21,10080.0,...,31763.0,False,NaN,NaN,algorithm_risk_score,False,False,False,True,False
5,5,app_275,NaN,***ler,***com,***912,***.70,F,***982,10019.0,...,49933.0,False,NaN,NaN,algorithm_risk_score,False,False,False,True,False
6,6,app_099,NaN,***ing,***com,***503,***.45,Male,***990,10022.0,...,30159.0,True,5.6,27000.0,NaN,False,False,False,True,False
7,7,app_246,NaN,***era,***com,***864,***.59,F,***-11,90223.0,...,21809.0,True,2.8,38000.0,NaN,False,False,False,True,False
8,8,app_042,NaN,***pez,***com,***530,***142,Male,***-04,10044.0,...,15974.0,False,NaN,NaN,algorithm_risk_score,False,False,True,False,False
9,9,app_348,NaN,***ell,***com,***400,***121,Male,***-10,10080.0,...,13794.0,False,NaN,NaN,insufficient_credit_history,False,False,False,True,False


### 1.4 Auditability & Provenance Finding

**Finding:** Missing `processing_timestamp` is a **material audit trail / provenance gap**.  
This should be treated as an **upstream instrumentation deficiency**, not a downstream cleaning bug.

In [7]:
ts_rows = extract_processing_timestamp_rows(dq_post)

# Show only the key columns if present
cols_preferred = [c for c in ["rule_id", "severity", "field_path", "description", "affected_count", "affected_pct", "n_rows"] if c in ts_rows.columns]
display(ts_rows[cols_preferred] if cols_preferred else ts_rows)

,rule_id,severity,field_path,description
0,R_APP_001,high,processing_timestamp,Missing or blank processing timestamp.


#### 1.4.1 Record-Keeping Evidence: Fairness Summary Artifact

The bias analysis consolidates fairness metrics and persists them as a reusable evidence artefact
(`data/quality/fairness_summary.csv`). This supports auditability by ensuring results are stored,
referenceable, and can be cited with concrete numbers across reports.

In [8]:
# Evidence artefact produced by the bias notebook
FAIRNESS_SUMMARY_PATH = paths.pii_inventory.parents[1] / "fairness_summary.csv"  
fairness_summary = pd.read_csv(FAIRNESS_SUMMARY_PATH)
display(fairness_summary.head(10))


,analysis,metric_value,four_fifths_flag,p_value,significant_at_05,note
0,Gender — Disparate Impact Ratio,0.7698,True,0.000826,True,Female rate 50.8% vs Male 66.0%
1,Gender — Demographic Parity Difference,-0.1519,NaN,0.000826,True,Negative = Female approval rate below Male
2,Age — DI ratio (<25 vs 25-34),0.9881,False,0.011477,True,n=13
3,Age — DI ratio (35-44 vs 25-34),1.4026,False,0.011477,True,n=174
4,Age — DI ratio (45-54 vs 25-34),1.3780,False,0.011477,True,n=87
5,Age — DI ratio (55-64 vs 25-34),1.3380,False,0.011477,True,n=56
6,Age — DI ratio (65+ vs 25-34),1.1528,False,0.011477,True,n=13
7,Interest Rate — Gender gap (approved only),Male=4.7000 Female=4.4000,NaN,0.326435,False,"n Male=163, n Female=127"


#### 1.4.2 Evidence Manifest (Hashes + Timestamps)

To strengthen auditability, we generate a small evidence manifest containing file paths, last-modified
timestamps, and SHA-256 hashes for the key artefacts used in the governance narrative.

In [9]:


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Compute SHA-256 hash for a file (audit-friendly fingerprint)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

evidence_files = [
    ("pii_inventory", paths.pii_inventory),
    ("applications_analysis", paths.applications_analysis),
    ("applications_curated_full", paths.applications_curated_full),
    ("dq_postclean_report", paths.dq_postclean),
    ("fairness_summary", FAIRNESS_SUMMARY_PATH),
]

rows = []
for name, p in evidence_files:
    if p.exists():
        rows.append({
            "artefact": name,
            "path": str(p),
            "last_modified": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
            "sha256": sha256_file(p),
        })
    else:
        rows.append({"artefact": name, "path": str(p), "last_modified": None, "sha256": None})

manifest = pd.DataFrame(rows)
display(manifest)

# Optional: persist manifest as part of the evidence pack
MANIFEST_PATH = paths.pii_inventory.parents[1] / "evidence_manifest.csv"  
manifest.to_csv(MANIFEST_PATH, index=False)

,artefact,path,last_modified,sha256
0,pii_inventory,C:\Users\anton\OneDrive - Nova SBE\Nova\S2\DEG...,2026-02-28T16:45:45,2e74b36bf781d57d8d4d0b0e43d56c07d81464a305ce06...
1,applications_analysis,C:\Users\anton\OneDrive - Nova SBE\Nova\S2\DEG...,2026-02-28T13:13:22,67c8f3649bcbbd6193de61c4ef69cee5d0dab08fe13a38...
2,applications_curated_full,C:\Users\anton\OneDrive - Nova SBE\Nova\S2\DEG...,2026-02-28T16:45:45,158b7d1e6a31c0c0b53f6c963df3396a10be67b6505135...
3,dq_postclean_report,C:\Users\anton\OneDrive - Nova SBE\Nova\S2\DEG...,2026-02-28T08:27:40,a852214eb7b32cf0d441d1fa241fe478631f55d4187b92...
4,fairness_summary,C:\Users\anton\OneDrive - Nova SBE\Nova\S2\DEG...,2026-03-01T14:47:40,daf260611e2aa22ec4f8c28dc211ea738d52d02a8fbdb4...


#### 1.4.3 Audit Scope & Missingness (Analysis Dataset)

Fairness metrics depend on group membership and outcomes. We quantify missingness in key fields to ensure
the scope of downstream analyses is transparent and reproducible.

In [10]:
key_cols = ["clean_loan_approved", "clean_gender", "age_band", "applicant_pseudo_id"]

scope = []
n = len(analysis)
for c in key_cols:
    if c in analysis.columns:
        missing = int(analysis[c].isna().sum())
        scope.append({"field": c, "missing_n": missing, "missing_pct": missing / n * 100})
    else:
        scope.append({"field": c, "missing_n": None, "missing_pct": None})

scope_df = pd.DataFrame(scope)
display(scope_df)

,field,missing_n,missing_pct
0,clean_loan_approved,0,0.0
1,clean_gender,3,0.6
2,age_band,5,1.0
3,applicant_pseudo_id,0,0.0


#### 1.4.4 Explainability Gap: Rejection Reason Opacity

A large share of rejections attributed to a generic or opaque reason (e.g., `algorithm_risk_score`) reduces
explainability and weakens the audit trail of adverse decisions. This is a governance risk independent of
group fairness metrics.

In [11]:
# Only run if rejection reason exists in the analysis dataset
if "clean_rejection_reason" in analysis.columns and "clean_loan_approved" in analysis.columns:
    rejected = analysis.loc[analysis["clean_loan_approved"].eq(False)].copy()

    if len(rejected) == 0:
        print("No rejected records found in analysis.")
    else:
        reason_counts = rejected["clean_rejection_reason"].fillna("MISSING").value_counts()
        reason_pct = (reason_counts / reason_counts.sum() * 100).round(2)

        reason_tbl = pd.DataFrame({"count": reason_counts, "pct": reason_pct})
        display(reason_tbl.head(15))

        # Highlight the concentration in algorithm_risk_score if present
        if "algorithm_risk_score" in reason_tbl.index:
            share = float(reason_tbl.loc["algorithm_risk_score", "pct"])
            print(f"Share of rejections labelled 'algorithm_risk_score': {share:.2f}%")
else:
    print("clean_rejection_reason or clean_loan_approved not available in analysis dataset.")

,count,pct
clean_rejection_reason,,
algorithm_risk_score,169,81.25
insufficient_credit_history,23,11.06
high_dti_ratio,12,5.77
low_income,4,1.92


Share of rejections labelled 'algorithm_risk_score': 81.25%


### 1.5 Evidence Summary 

In [12]:
summary = f"""
Privacy controls evidence:

- PII inventory identifies {len(pii_fields)} PII fields and {len(quasi_pii_fields)} quasi-identifiers/proxy candidates.
- Data minimisation control passed: applications_analysis.csv contains 0 PII fields (enforced via assertion).
- Layer separation is maintained: curated_full may include PII and must remain access-restricted (least privilege).
- Pseudonymisation is used for linkage: applicant_pseudo_id enables deterministic joins without exposing identifiers.
- Auditability gap: missing processing_timestamp remains a material provenance issue and should be treated as an upstream instrumentation/control deficiency.
""".strip()

print(summary)

Privacy controls evidence:

- PII inventory identifies 5 PII fields and 4 quasi-identifiers/proxy candidates.
- Data minimisation control passed: applications_analysis.csv contains 0 PII fields (enforced via assertion).
- Layer separation is maintained: curated_full may include PII and must remain access-restricted (least privilege).
- Pseudonymisation is used for linkage: applicant_pseudo_id enables deterministic joins without exposing identifiers.
- Auditability gap: missing processing_timestamp remains a material provenance issue and should be treated as an upstream instrumentation/control deficiency.


## 3 Residual Risks 